# WESAD HSSL+DPBL Stress Detection Pipeline â€” Google Colab

**Uses T4 GPU, auto-saves checkpoints to Google Drive.**

---
### Setup Awal
1. Runtime â†’ Change runtime type â†’ **T4 GPU**
2. Jalankan cell dari atas ke bawah secara berurutan
3. Jika runtime terputus (batas 12 jam), **mount ulang Drive** â†’ **Run All** â†’ checkpoint otomatis di-skip

---

## Cell 0 â€” Mount Google Drive & Clone Repo

In [ ]:
import os, sys, subprocess

WORK = "/content"
os.chdir(WORK)

# Mount Google Drive untuk checkpoint persisten
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
if not os.path.exists("/content/hssl-plus-dpbl"):
    !git clone https://github.com/Notdet3cted/hssl-plus-dpbl.git

%cd /content/hssl-plus-dpbl

# Buat folder Drive untuk checkpoint (sekali)
DRIVE_CKPT = "/content/drive/MyDrive/hssl_dpbl_checkpoints"
!mkdir -p "{DRIVE_CKPT}"

# Backup checkpoints/config ke Drive supaya survive session restart
# Symlink tidak bisa di Colab, jadi kita copy manual nanti di setiap stage
print("Setup selesai. Semua checkpoint akan di-backup ke Drive.")


## Cell 1 â€” Install Dependencies & Setup Paths

In [ ]:
# Install requirements
!pip install -r requirements.txt -q

!mkdir -p /content/drive/MyDrive/hssl_dpbl_results
print("Dependencies installed, results folder created.")


## Cell 2 â€” Download Dataset WESAD

Pilih **salah satu** metode di bawah.

---
### Opsi A: Download dari Kaggle (via Kaggle API)
Butuh Kaggle API key (`kaggle.json`). Upload ke Drive dulu.

### Opsi B: Upload ZIP WESAD ke Drive
1. Download WESAD dari https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx
2. Upload ZIP ke `/content/drive/MyDrive/wesad.zip`
3. Jalankan cell Opsi B

### Opsi C: Dataset sudah ada di Drive sebelumnya
Lewati cell ini, edit `config.yaml` â†’ `raw_data:` diarahkan ke path Drive.

In [ ]:
# Pilih metode:
METHOD = "B"  # Ganti ke "A" atau "C" sesuai kebutuhan

if METHOD == "A":
    # === Opsi A: Kaggle API ===
    !mkdir -p ~/.kaggle
    !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d datasets/orvile/wesad-wearable-stress-affect-detection-dataset
    !unzip -q wesad-wearable-stress-affect-detection-dataset.zip -d data/raw/

elif METHOD == "B":
    # === Opsi B: ZIP dari Drive ===
    ZIP_PATH = "/content/drive/MyDrive/wesad.zip"
    if os.path.exists(ZIP_PATH):
        !unzip -q "{ZIP_PATH}" -d data/raw/
        print("Dataset extracted from Drive.")
    else:
        print(f"FILE TIDAK DITEMUKAN: {ZIP_PATH}")
        print("Upload wesad.zip ke Google Drive dulu.")
        print("Hentikan eksekusi — dataset diperlukan.")
        raise SystemExit(1)

elif METHOD == "C":
    print("Dataset sudah ada â€” pastikan config.yaml raw_data sudah benar.")

print("Dataset ready.")



---
## Cell 3 â€” Stage 1: Data Preparation

*Acquisition â†’ Preprocessing â†’ LOSO folds â†’ Windowing*

Skips jika output sudah ada.

In [ ]:
!python run_colab.py --mode data_prep --skip-setup
# Backup semua data ke Drive
!cp -r /content/drive/MyDrive/hssl_dpbl_data/processed/* /content/drive/MyDrive/hssl_dpbl_data/processed/ 2>/dev/null || true
!mkdir -p /content/drive/MyDrive/hssl_dpbl_logs /content/drive/MyDrive/hssl_dpbl_experiments
!cp -r logs/* /content/drive/MyDrive/hssl_dpbl_logs/ 2>/dev/null || true



---
## Cell 4 â€” Stage 2: SSL Pre-training & Embeddings

*Pre-train SimSiam SSL encoder, generate SSL embeddings.*
~1-2 jam di T4.

In [ ]:
!python run_colab.py --mode server_ssl --ssl_epochs 30 --skip-setup

# Backup checkpoints ke Drive
!cp -r checkpoints/* /content/drive/MyDrive/hssl_dpbl_checkpoints/ 2>/dev/null || true



---
## Cell 5 â€” Stage 3: Baseline Classifiers (RF, CNN, SSL)

In [ ]:
!python run_colab.py --mode server_a --epochs 20 --skip-setup



---
## Cell 6 â€” Stage 4: HSSL Pre-training, Embeddings & Classifier

In [ ]:
!python run_colab.py --mode server_b --epochs 20 --skip-setup

# Backup checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/hssl_dpbl_checkpoints/ 2>/dev/null || true



---
## Cell 7 â€” Stage 5: DPBL Training, Embeddings & HSSL+DPBL / SSL+DPBL Classifiers

Requires HSSL checkpoints dari Stage 4.

In [ ]:
# Restore checkpoints from Drive jika perlu (misal runtime restart)
!cp -r /content/drive/MyDrive/hssl_dpbl_checkpoints/* checkpoints/ 2>/dev/null || true

!python run_colab.py --mode server_c --epochs 20 --skip-setup

# Backup
!cp -r checkpoints/* /content/drive/MyDrive/hssl_dpbl_checkpoints/ 2>/dev/null || true



---
## Cell 8 â€” Stage 6: Robustness Testing

*Reduced to 5 iterations for Colab speed.*

In [ ]:
!python run_colab.py --mode server_d --robust_iter 5 --epochs 20 --skip-setup



---
## Cell 9 â€” Stage 7: Evaluation, Stats & Dashboard

Hasil akhir: `interactive_dashboard.html`

In [ ]:
!python run_colab.py --mode eval --skip-setup

# Copy semua hasil ke Drive
!cp -r results/* /content/drive/MyDrive/hssl_dpbl_results/ 2>/dev/null || true
!cp -r dashboard/* /content/drive/MyDrive/hssl_dpbl_results/ 2>/dev/null || true
!cp -r experiments/* /content/drive/MyDrive/hssl_dpbl_experiments/ 2>/dev/null || true
!cp -r logs/* /content/drive/MyDrive/hssl_dpbl_logs/ 2>/dev/null || true



---
## Hasil

Semua output tersimpan di **Google Drive**:
| Folder | Isi |
|--------|-----|
| `MyDrive/hssl_dpbl_checkpoints/` | Model weights (HSSL, DPBL, SSL) â†’ survive session restart |
| `MyDrive/hssl_dpbl_results/` | All CSV/JSON metrics + **interactive_dashboard.html** |

### Jika Runtime Terputus
1. Ulangi dari **Cell 0** (mount Drive)
2. **Run All** â†’ Stage yang sudah selesai akan skip otomatis (`--skip_existing`)
3. Checkpoints dipulihkan dari Drive di cell DPBL

---

### Troubleshooting

| Problem | Solution |
|---------|----------|
| GPU memory OOM | Edit `config.yaml` â†’ kurangi `batch_size: 64` atau `32` |
| Runtime disconnected | Ses 1: jalankan stage 1-4. Ses 2: Run All dari cell 0 (skip otomatis) |
| Dataset not found | Ganti metode download di Cell 2 atau upload manual ke Drive |
| Error `no module named ...` | Jalankan ulang Cell 1 (`!pip install -r requirements.txt`) |